<a href="https://colab.research.google.com/github/JiHyeonKu/JeonGiPeu/blob/main/ConvenienceStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import re
import sys
import os
from datetime import datetime

# ── Windows 터미널 UTF-8 강제 적용 (§2.1) ────────────────────
if sys.platform == "win32":
    sys.stdout.reconfigure(encoding="utf-8")
    sys.stderr.reconfigure(encoding="utf-8")
    sys.stdin.reconfigure(encoding="utf-8")

# ── 파일 경로 상수 ────────────────────────────────────────────
PRODUCT_DATA  = "productData.txt"
PRODUCT_LOG   = "productLog.txt"
CUSTOMER_DATA = "customerData.txt"
CUSTOMER_LOG  = "customerLog.txt"
FILES = [PRODUCT_DATA, PRODUCT_LOG, CUSTOMER_DATA, CUSTOMER_LOG]

# 프로그램 전역 상태
current_datetime: datetime | None = None   # 6.1.4 현재 일시

# ══════════════════════════════════════════════════════════════
#  SECTION 1 : 파일 I/O 헬퍼
# ══════════════════════════════════════════════════════════════
def read_file(path: str) -> list[str]:
    """파일을 읽어 줄 리스트로 반환 (빈 줄 제외)"""
    if not os.path.exists(path):
        print(f"{path} 파일이 존재하지 않습니다.")
        sys.exit(1)
    with open(path, "r", encoding="utf-8", newline="") as f:
        return [line.rstrip("\r\n") for line in f if line.strip()]

def write_file(path: str, lines: list[str]) -> None:
    """lines를 파일에 덮어씀"""
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lines) + ("\n" if lines else ""))

def append_line(path: str, line: str) -> None:
    with open(path, "a+", encoding="utf-8") as f:
        f.seek(0, os.SEEK_END)
        if f.tell() > 0:
            f.seek(f.tell() - 1)
            if f.read(1) != "\n":
                f.write("\n")
        f.write(line + "\n")

def parse_record(line: str) -> list[str]:
    """파이프(|) 구분 레코드를 필드 리스트로 파싱, 앞뒤 공백 제거"""
    return [field.strip() for field in line.split("|")]

# ══════════════════════════════════════════════════════════════
#  SECTION 2 : 유효성 검사 (4절 데이터 요소)
# ══════════════════════════════════════════════════════════════
DATETIME_RE = re.compile(
    r"^\d{4}-(0[1-9]|1[0-2])-(0[1-9]|[12]\d|3[01])"
    r"\s([01]\d|2[0-3]):[0-5]\d:[0-5]\d$"
)
CATEGORIES      = {"냉동식품", "냉장식품", "과자", "음료", "주류", "생활용품", "기타"}
PAYMENT_METHODS = {"현금", "카드", "포인트"}
EVENT_TYPES     = {"IN", "OUT", "DISPOSE"}
CUSTOMER_NAME_RE = re.compile(r"^([가-힣]{2,5}|[a-zA-Z]{3,20})$")
EVENT_ID_RE      = re.compile(r"^[1-9][0-9]*$")
PURCHASE_ID_RE   = re.compile(r"^[1-9][0-9]*$")
PRODUCT_ID_RE    = re.compile(r"^\d{4}$")
CUSTOMER_ID_RE   = re.compile(r"^\d{4}$")

def is_valid_datetime(s: str) -> bool:
    if not DATETIME_RE.match(s):
        return False
    try:
        datetime.strptime(s, "%Y-%m-%d %H:%M:%S")
        return True
    except ValueError:
        return False

def parse_dt(s: str) -> datetime:
    return datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

def is_valid_product_id(s: str) -> bool:
    return bool(PRODUCT_ID_RE.match(s))

def validate_event_time(pid_list, event_time):
    # 이벤트 시각 검증:
    # - 현재 일시보다 미래일 수 없음
    # - 상품의 입고일보다 이전일 수 없음

    products = _load_products()

    if current_datetime is None:
        print("현재 일시가 설정되지 않았습니다.")
        return False

    if event_time > current_datetime:
        print("이벤트 시각은 현재보다 미래일 수 없습니다.")
        return False

    for pid in pid_list:
        if pid not in products:
            print("존재하지 않는 상품입니다.")
            return False

        in_date = parse_dt(products[pid]["in_date"])
        if event_time < in_date:
            print("이벤트 시각은 입고일보다 이전일 수 없습니다.")
            return False

    return True

def is_valid_customer_id(s: str) -> bool:
    return bool(CUSTOMER_ID_RE.match(s))

def is_valid_category(s: str) -> bool:
    return s in CATEGORIES

def is_valid_price(s: str) -> bool:
    return s.isdigit()   # 0 이상의 정수

def is_valid_customer_name(s: str) -> bool:
    return bool(CUSTOMER_NAME_RE.match(s))

def is_valid_event_id(s: str) -> bool:
    return bool(EVENT_ID_RE.match(s))

# ══════════════════════════════════════════════════════════════
#  SECTION 3 : 재고 계산 (이벤트 로그 기반)
# ══════════════════════════════════════════════════════════════
def calc_stock(product_id: str, as_of: datetime | None = None) -> int:
    """
    productLog.txt를 읽어 product_id 의 현재 재고를 계산.
    as_of 가 주어지면 해당 시각 이전 이벤트만 반영 (6.6 가상 일시 조회용).
    재고 = 이벤트 로그(productLog.txt) 기반으로 계산 (IN - OUT - DISPOSE)
    """
    stock = 0
    for line in read_file(PRODUCT_LOG):
        fields = parse_record(line)
        # 레코드 형식: <이벤트ID>|<상품ID(들)>|<이벤트종류>|<수량>|<이벤트시각>
        if len(fields) != 5:
            continue
        event_id, ids_raw, etype, qty_s, evt_time_s = fields
        ids = [i.strip() for i in ids_raw.split(",")]
        if product_id not in ids:
            continue
        if as_of and parse_dt(evt_time_s) > as_of:
            continue
        if etype == "IN":
            stock += ids.count(product_id)
        elif etype in ("OUT", "DISPOSE"):
            # OUT/DISPOSE 레코드의 수량은 포함된 상품ID 총 개수이므로
            # 해당 상품ID가 포함된 경우 1씩 차감
            stock -= ids.count(product_id)
    return stock

def get_all_stocks(as_of: datetime | None = None) -> dict[str, int]:
    """모든 상품ID → 재고 수량 딕셔너리 반환"""
    product_ids = {parse_record(l)[0] for l in read_file(PRODUCT_DATA)}
    return {pid: calc_stock(pid, as_of) for pid in product_ids}

def next_event_id() -> str:
    """productLog.txt 에서 최대 이벤트ID + 1 반환"""
    max_id = 0
    for line in read_file(PRODUCT_LOG):
        fields = parse_record(line)
        if fields and EVENT_ID_RE.match(fields[0]):
            max_id = max(max_id, int(fields[0]))
    return str(max_id + 1)

def next_purchase_id() -> str:
    """customerLog.txt 에서 최대 구매ID + 1 반환"""
    max_id = 0
    for line in read_file(CUSTOMER_LOG):
        fields = parse_record(line)
        if fields and PURCHASE_ID_RE.match(fields[0]):
            max_id = max(max_id, int(fields[0]))
    return str(max_id + 1)

def next_product_id() -> str:
    """productData.txt 에서 최대 상품ID + 1 을 4자리로 반환"""
    max_id = 0
    for line in read_file(PRODUCT_DATA):
        fields = parse_record(line)
        if fields and PRODUCT_ID_RE.match(fields[0]):
            max_id = max(max_id, int(fields[0]))
    return str(max_id + 1).zfill(4)

def next_customer_id() -> str:
    """customerData.txt 에서 최대 고객ID + 1 을 4자리로 반환 (0000 제외)"""
    max_id = 0
    for line in read_file(CUSTOMER_DATA):
        fields = parse_record(line)
        if fields and CUSTOMER_ID_RE.match(fields[0]) and fields[0] != "0000":
            max_id = max(max_id, int(fields[0]))
    return str(max_id + 1).zfill(4)

def last_event_time() -> datetime | None:
    """productLog + customerLog 에서 가장 마지막 이벤트 시각 반환"""
    times = []
    for path in (PRODUCT_LOG, CUSTOMER_LOG):
        for line in read_file(path):
            fields = parse_record(line)
            if len(fields) >= 5 and is_valid_datetime(fields[-1]):
                times.append(parse_dt(fields[-1]))
    return max(times) if times else None

# ══════════════════════════════════════════════════════════════
#  SECTION 4 : 무결성 검사 (5.9절)
# ══════════════════════════════════════════════════════════════
def check_integrity() -> None:
    """
    문법·의미·파일 간 무결성 규칙 검사.
    위반 레코드 발견 시 출력 후 sys.exit().
    """
    print("무결성 검사 중… ", end="", flush=True)
    errors = []

    # --- productData.txt 검사 ---
    product_ids = set()
    products = {}   # id -> {name, price, category, in_date, exp_date}
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        ok = (
            len(f) == 6
            and is_valid_product_id(f[0])
            and len(f[1]) >= 1
            and is_valid_price(f[2])
            and is_valid_category(f[3])
            and is_valid_datetime(f[4])
            and is_valid_datetime(f[5])
            and parse_dt(f[4]) <= parse_dt(f[5])   # 입고일 <= 유통기한
        )
        if not ok:
            errors.append(f"<{PRODUCT_DATA}> => {line}")
        else:
            if f[0] in product_ids:
                errors.append(f"<{PRODUCT_DATA}> 중복 상품ID => {line}")
            product_ids.add(f[0])
            products[f[0]] = {"name": f[1], "price": int(f[2]),
                              "category": f[3], "in_date": f[4], "exp_date": f[5]}

    # --- productLog.txt 검사 ---
    event_ids = set()
    for line in read_file(PRODUCT_LOG):
        f = parse_record(line)
        ok = (
            len(f) == 5
            and is_valid_event_id(f[0])
            and f[2] in EVENT_TYPES
            and f[3].isdigit() and int(f[3]) >= 1
            and is_valid_datetime(f[4])
        )
        if ok:
            ids = [i.strip() for i in f[1].split(",")]
            if any(pid not in product_ids for pid in ids):
                ok = False
            if int(f[3]) != len(ids):
                ok = False
        if not ok:
            errors.append(f"<{PRODUCT_LOG}> => {line}")
        elif f[0] in event_ids:
            errors.append(f"<{PRODUCT_LOG}> 중복 이벤트ID => {line}")
        else:
            event_ids.add(f[0])

    # --- customerData.txt 검사 ---
    customer_ids = set()
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        ok = (
            len(f) == 3
            and is_valid_customer_id(f[0])
            and is_valid_customer_name(f[1])
            and f[2].isdigit()
        )
        if not ok:
            errors.append(f"<{CUSTOMER_DATA}> => {line}")
        else:
            customer_ids.add(f[0])

    # --- customerLog.txt 검사 ---
    purchase_ids = set()
    for line in read_file(CUSTOMER_LOG):
        f = parse_record(line)
        ok = (
            len(f) == 7
            and PURCHASE_ID_RE.match(f[0])
            and is_valid_customer_id(f[1])
            and (f[1] == "0000" or f[1] in customer_ids)
            and is_valid_price(f[3])
            and f[4] in PAYMENT_METHODS
            and f[5].isdigit() and int(f[5]) >= 1
            and is_valid_datetime(f[6])
        )
        if ok:
            ids = [i.strip() for i in f[2].split(",")]
            if any(pid not in product_ids for pid in ids):
                ok = False
            if int(f[5]) != len(ids):
                ok = False
        if not ok:
            errors.append(f"<{CUSTOMER_LOG}> => {line}")
        elif f[0] in purchase_ids:
            errors.append(f"<{CUSTOMER_LOG}> 중복 구매ID => {line}")
        else:
            purchase_ids.add(f[0])

    if errors:
        print("데이터 오류가 발생했습니다.")
        for e in errors:
            print(e)
        sys.exit(1)
    else:
        print("완료")

# ══════════════════════════════════════════════════════════════
#  SECTION 5 : 초기 설정 (6.1절)
# ══════════════════════════════════════════════════════════════
def startup_banner() -> None:
    """6.1.1 초기 화면 출력"""
    print("=" * 60)
    print("'스마트 무인 편의점' 재고 관리 프로그램 v1.0")
    print("=" * 60)

def check_files() -> None:
    """6.1.2 필수 데이터 파일 존재 여부 및 읽기/쓰기 권한 확인 (없으면 종료)"""
    for f in FILES:
        # Create file if it doesn't exist
        if not os.path.exists(f):
            print("필수 파일이 존재하지 않습니다.")
            sys.exit(1)
        if not os.access(f, os.R_OK | os.W_OK):
            print("파일 접근 권한이 없습니다.")
            sys.exit(1)

def input_current_datetime() -> datetime:
    """6.1.4 현재 일시 입력"""
    last = last_event_time()
    while True:
        print()
        s = input("현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):\n> ").strip()
        print()
        if not is_valid_datetime(s):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-04-02 14:00:00)")
            continue
        dt = parse_dt(s)
        if last and dt <= last:
            print("현재 일시는 기존 로그의 마지막 시각보다 늦어야 합니다.")
            continue
        print("기준 시각이 설정되었습니다.")
        return dt

def auto_dispose(now: datetime) -> None:
    """6.1.5 유통기한 경과 상품을 자동으로 폐기 이벤트로 기록"""
    expired = []   # (pid, stock) 쌍
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6 and parse_dt(f[5]) < now:  # 유통기한은 판매 가능한 마지막 시각 → 현재 시각이 유통기한보다 이후일 경우 폐기
            pid = f[0]
            stock = calc_stock(pid)
            if stock >= 1:
                expired.append((pid, stock))

    if not expired:
        print("현재 폐기 처리할 유통기한 경과 상품이 없습니다.")
        return

    eid = next_event_id()
    ids_list = []
    for pid, stock in expired:
      ids_list.extend([pid] * stock)

    ids_field = ", ".join(ids_list)
    total_qty = len(ids_list)
    time_s = now.strftime("%Y-%m-%d %H:%M:%S")
    record = f"{eid}|{ids_field}|DISPOSE|{total_qty}|{time_s}"
    try:
        append_line(PRODUCT_LOG, record)
    except OSError:
        print("데이터 파일 접근에 실패하여 폐기 처리를 중단합니다.")
        sys.exit(1)
    print(f"총 {total_qty}개의 재고가 유통기한 경과로 자동 폐기 처리되었습니다.")

# ══════════════════════════════════════════════════════════════
#  SECTION 6 : 메인 메뉴 (6.2절)
# ══════════════════════════════════════════════════════════════
def main_menu() -> None:
    """6.2 메인 메뉴 루프"""
    while True:
        print("\n1. 고객 관리")
        print("2. 재고 관리")
        print("3. 판매 처리")
        print("4. 가상의 일시 기준 재고 조회")
        print("0. 종료")
        choice = input("메뉴 > ").strip()
        if not choice.isdigit():
            print("숫자를 입력해주세요.")
            continue
        match choice:
            case "1": menu_customer()
            case "2": menu_inventory()
            case "3": menu_sales()
            case "4": menu_virtual_stock()
            case "0":
                print("프로그램을 종료합니다.")
                sys.exit(0)
            case _:
                print("올바른 메뉴 번호를 입력해주세요.")
                continue
        # 기능 수행 후 현재 일시 재입력 (6.1.4)
        global current_datetime
        current_datetime = input_current_datetime()
        auto_dispose(current_datetime)

# ══════════════════════════════════════════════════════════════
#  SECTION 7 : 고객 관리 (6.3절)
# ══════════════════════════════════════════════════════════════
def menu_customer() -> None:
    """6.3 고객 관리 서브 메뉴"""
    while True:
        print("\n1. 고객 등록\n2. 고객 조회\n0. 메인 메뉴")
        choice = input("메뉴 > ").strip()
        if not choice.isdigit():
            print("숫자를 입력해주세요.")
            continue
        match choice:
            case "1": register_customer(); return
            case "2": search_customer(); return
            case "0": return
            case _:
                print("올바른 메뉴 번호를 입력해주세요.")

def register_customer() -> None:
    """6.3.3 고객 등록"""
    while True:
        name = input("고객명: ").strip()
        if not is_valid_customer_name(name):
            print("고객명은 한글 2~5자 또는 영문 3~20자여야 합니다.")
            continue
        break
    new_id = next_customer_id()
    print(f"\n고객ID: {new_id}  고객명: {name}  초기 포인트: 0")
    while True:
        confirm = input("등록하시겠습니까? (Y/N): ").strip()
        if confirm in ("Y", "y"):
            append_line(CUSTOMER_DATA, f"{new_id}|{name}|0")
            print("고객 등록이 완료되었습니다.")
            return
        elif confirm in ("N", "n"):
            print("등록을 취소했습니다.")
            return
        else:
            print("Y 또는 N으로 입력해 주세요.")

def search_customer() -> None:
    """6.3.4 고객 조회"""
    while True:
        raw = input("검색 (예: 1 1023 또는 2 김철수): ").strip()
        parts = raw.split(" ", 1)
        if len(parts) < 2 or parts[0] not in ("1", "2"):
            print("형식에 맞게 입력해 주세요. (예: 1 1023 또는 2 김철수)")
            continue
        criterion, keyword = parts[0], parts[1].strip()
        break
    results = []
    for line in read_file(CUSTOMER_DATA):
        f = parse_record(line)
        if len(f) != 3:
            continue
        if criterion == "1" and f[0] == keyword:
            results.append(f)
        elif criterion == "2" and f[1] == keyword:
            results.append(f)
    if not results:
        print("검색 결과가 없습니다.")
        return
    # 6.3.5 고객 정보 출력 (고객ID 오름차순)
    results.sort(key=lambda x: x[0])
    print(f"\n{'고객ID':<8} {'고객명':<12} {'포인트'}")
    print("-" * 32)
    for f in results:
        print(f"{f[0]:<8} {f[1]:<12} {f[2]}")
    print(f"\n총 {len(results)}건 검색됨")

# ══════════════════════════════════════════════════════════════
#  SECTION 8 : 재고 관리 (6.4절)
# ══════════════════════════════════════════════════════════════
def menu_inventory() -> None:
    """6.4 재고 관리 서브 메뉴"""
    while True:
        print("\n1. 상품 입고\n2. 재고 조회\n3. 전체 재고 출력\n4. 폐기 상품 조회\n0. 메인 메뉴")
        choice = input("메뉴 > ").strip()
        if not choice.isdigit():
            print("숫자를 입력해주세요.")
            continue
        match choice:
            case "1": receive_product(); return
            case "2": search_inventory(); return
            case "3": print_all_inventory(); return
            case "4": search_disposed(); return
            case "0": return
            case _:
                print("올바른 메뉴 번호를 입력해주세요.")

def receive_product() -> None:
    """6.4.3 상품 입고"""
    # 상품명 입력
    while True:
        name = input("상품명: ").strip()
        if not name or "\\" in name or "/" in name:
            print("상품명은 1글자 이상이어야 하며 '\\'와 '/'를 포함할 수 없습니다.")
            continue
        break
    # 카테고리 입력
    while True:
        cat = input(f"카테고리 {sorted(CATEGORIES)}: ").strip()
        if not is_valid_category(cat):
            print("정확한 카테고리 문자열 중 하나를 입력해 주세요.")
            continue
        break
    # 가격 입력
    while True:
        price_s = input("상품 가격: ").strip()
        if not is_valid_price(price_s):
            print("상품 가격은 0 이상의 정수여야 합니다.")
            continue
        break
    # 입고일 입력
    while True:
        in_date = input("입고일 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(in_date):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-03-25 14:00:00)")
            continue
        if parse_dt(in_date) > current_datetime:
            print("입고일은 현재 일시 이전이어야 합니다.")
            continue
        break

    # 유통기한 입력
    while True:
        exp_date = input("유통기한 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(exp_date):
            print("날짜 형식이 올바르지 않습니다. (예: 2026-06-30 23:59:59)")
            continue
        if parse_dt(exp_date) <= parse_dt(in_date):
            print("유통기한은 입고일 이후여야 합니다.")
            continue
        break
    new_pid = next_product_id()

    while True:
        qty_s = input("입고 수량: ").strip()
        if not qty_s.isdigit() or int(qty_s) <= 0:
            print("수량은 1 이상의 정수로 입력하세요.")
            continue
        qty = int(qty_s)
        break

    # 🔥 전체 정보 출력
    print(f"""
    상품ID: {new_pid}
    상품명: {name}
    카테고리: {cat}
    가격: {price_s}
    입고일: {in_date}
    유통기한: {exp_date}
    수량: {qty}
    """)

    while True:
        confirm = input("입고하시겠습니까? (Y/N): ").strip()
        if confirm in ("Y", "y"):
            append_line(PRODUCT_DATA, f"{new_pid}|{name}|{price_s}|{cat}|{in_date}|{exp_date}")

            eid = next_event_id()
            time_s = current_datetime.strftime("%Y-%m-%d %H:%M:%S")

            ids = [new_pid] * qty
            ids_field = ", ".join(ids)

            if not validate_event_time(ids, current_datetime):
                return

            append_line(PRODUCT_LOG, f"{eid}|{ids_field}|IN|{qty}|{time_s}")

            print("상품 입고가 완료되었습니다.")
            return

        elif confirm in ("N", "n"):
            print("입고를 취소했습니다.")
            return

        else:
            print("Y 또는 N으로 입력해 주세요.")

def _load_products() -> dict:
    """productData.txt → {pid: {name, price, category, in_date, exp_date}}"""
    products = {}
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6:
            products[f[0]] = {
                "name": f[1], "price": int(f[2]),
                "category": f[3], "in_date": f[4], "exp_date": f[5]
            }
    return products

def search_inventory() -> None:
    """6.4.4 재고 조회"""
    criteria_map = {"1": "상품ID", "2": "상품명", "3": "상품 가격",
                    "4": "카테고리", "5": "입고일", "6": "유통기한"}
    while True:
        raw = input("검색 기준번호 검색어 (예: 2 콜라): ").strip()
        parts = raw.split(" ", 1)
        if len(parts) < 2 or parts[0] not in criteria_map:
            print("형식에 맞게 입력해 주세요. (예: 2 콜라)")
            continue
        crit, keyword = parts[0], parts[1].strip()
        break
    products = _load_products()
    stocks = get_all_stocks()
    matched = []
    for pid, p in products.items():
        if stocks.get(pid, 0) < 1:
            continue
        match crit:
            case "1":   hit = pid == keyword
            case "2":   hit = keyword in p["name"]
            case "3":   hit = _price_filter(keyword, p["price"])
            case "4":   hit = p["category"] == keyword
            case "5":   hit = _date_range_filter(keyword, p["in_date"])
            case "6":   hit = _date_range_filter(keyword, p["exp_date"])
            case _:
                hit = False
        if hit:
            matched.append((pid, p, stocks[pid]))
    if not matched:
        print("검색 결과가 없습니다.")
        return
    matched.sort(key=lambda x: x[0])
    _print_inventory_table(matched)

def _price_filter(keyword: str, price: int) -> bool:
    """가격 범위 검색 헬퍼 (예: '1000~3000', '2000 이하', '500 이상')"""
    keyword = keyword.replace(" ", "")
    if "~" in keyword:
        lo, hi = keyword.split("~", 1)
        return int(lo) <= price <= int(hi)
    if keyword.endswith("이하"):
        return price <= int(keyword[:-2])
    if keyword.endswith("이상"):
        return price >= int(keyword[:-2])
    return str(price) == keyword

def _date_range_filter(keyword: str, date_s: str) -> bool:
    """날짜 부분 문자열 / 범위 검색 헬퍼"""
    if "~" in keyword:
        parts = keyword.split("~", 1)
        lo = parts[0].strip() or "0000-00-00 00:00:00"
        hi = parts[1].strip() or "9999-12-31 23:59:59"
        return lo <= date_s <= hi
    return keyword in date_s

def _print_inventory_table(items: list) -> None:
    """재고 목록 출력 (6.4.5)"""
    print(f"\n{'상품ID':<8}{'상품명':<16}{'카테고리':<12}{'가격':>8}{'수량':>6}  {'입고일':<22}{'유통기한'}")
    print("-" * 90)
    for pid, p, qty in items:
        print(f"{pid:<8}{p['name']:<16}{p['category']:<12}{p['price']:>8}{qty:>6}"
              f"  {p['in_date']:<22}{p['exp_date']}")
    print(f"\n총 {len(items)}개 상품")

def print_all_inventory() -> None:
    """6.4.6 전체 재고 출력"""
    products = _load_products()
    stocks = get_all_stocks()
    in_stock = [(pid, p, stocks.get(pid, 0)) for pid, p in products.items()
                if stocks.get(pid, 0) >= 1]
    if not in_stock:
        print("등록된 상품이 없습니다.")
        return
    in_stock.sort(key=lambda x: x[0])
    _print_inventory_table(in_stock)
    print(f"총 {len(in_stock)}종  총 수량 {sum(q for _, _, q in in_stock)}개")

def search_disposed() -> None:
    """6.4.7 폐기 상품 조회"""
    products = _load_products()
    disposed = []
    for line in read_file(PRODUCT_LOG):
        f = parse_record(line)
        if len(f) == 5 and f[2] == "DISPOSE":
            disposed.append(f)
    if not disposed:
        print("폐기 처리된 상품 내역이 존재하지 않습니다.")
        return
    disposed.sort(key=lambda x: x[4], reverse=True)   # 이벤트 시각 내림차순
    print(f"\n{'이벤트ID':<12}{'상품명':<30}{'폐기수량':>8}  {'이벤트 시각'}")
    print("-" * 72)
    for f in disposed:
        names = ", ".join(products.get(pid.strip(), {}).get("name", "?")
                          for pid in f[1].split(","))
        print(f"{f[0]:<12}{names:<30}{f[3]:>8}  {f[4]}")

# ══════════════════════════════════════════════════════════════
#  SECTION 9 : 판매 처리 (6.5절)
# ══════════════════════════════════════════════════════════════
def menu_sales() -> None:
    """6.5 판매 처리"""

    print("\n[판매 처리]")

    raw = input("상품ID 입력 (예: 1001,1002,1003): ").strip()
    product_ids = [p.strip() for p in raw.split(",") if p.strip()]

    if not product_ids:
        print("상품ID를 입력해야 합니다.")
        return

    if len(product_ids) != len(set(product_ids)):
        print("같은 상품ID를 중복 입력할 수 없습니다.")
        return

    product_map = {}
    for line in read_file(PRODUCT_DATA):
        f = parse_record(line)
        if len(f) == 6:
            product_map[f[0]] = f

    for pid in product_ids:
        if not is_valid_product_id(pid):
            print(f"상품ID 형식 오류: {pid}")
            return

        if pid not in product_map:
            print(f"존재하지 않는 상품: {pid}")
            return

        stock = calc_stock(pid)
        if stock <= 0:
            print(f"{pid} 재고 없음")
            return

        exp_date = parse_dt(product_map[pid][5])
        if exp_date < current_datetime:
            print(f"{pid} 유통기한 지난 상품")
            return

    quantity = len(product_ids)

    customer_id = input("고객ID 입력 (비회원: 0000): ").strip()

    if not is_valid_customer_id(customer_id):
        print("고객ID 형식 오류")
        return

    customer_ids = {parse_record(l)[0] for l in read_file(CUSTOMER_DATA)}
    if customer_id != "0000" and customer_id not in customer_ids:
        print("존재하지 않는 고객")
        return

    print("결제 방식 선택: 1.현금  2.카드  3.포인트")
    pay_map = {"1": "현금", "2": "카드", "3": "포인트"}

    choice = input("선택: ").strip()
    if choice not in pay_map:
        print("잘못된 선택")
        return

    payment = pay_map[choice]

    if customer_id == "0000" and payment == "포인트":
        print("비회원은 포인트 결제 불가")
        return

    total_price = sum(int(product_map[pid][2]) for pid in product_ids)

    print(f"총 결제 금액: {total_price}원")

    confirm = input("결제하시겠습니까? (Y/N): ").strip()
    if confirm not in ("Y", "y"):
        print("판매 취소")
        return

    if not validate_event_time(product_ids, current_datetime):
        return

    eid = next_event_id()
    ids_field = ", ".join(product_ids)
    time_s = current_datetime.strftime("%Y-%m-%d %H:%M:%S")

    append_line(PRODUCT_LOG, f"{eid}|{ids_field}|OUT|{quantity}|{time_s}")

    purchase_id = next_purchase_id()

    append_line(
        CUSTOMER_LOG,
        f"{purchase_id}|{customer_id}|{ids_field}|{total_price}|{payment}|{quantity}|{time_s}"
    )

    print("판매 완료")
    return


# ══════════════════════════════════════════════════════════════
#  SECTION 10 : 가상의 일시 기준 재고 조회 (6.6절)
# ══════════════════════════════════════════════════════════════
def menu_virtual_stock() -> None:
    """6.6 가상의 일시 기준 재고 조회"""
    while True:
        s = input("가상의 일시 입력 (YYYY-MM-DD HH:MM:SS): ").strip()
        if not is_valid_datetime(s):
            print("올바른 날짜 형식이 아닙니다.")
            continue
        virtual_dt = parse_dt(s)
        break
    products = _load_products()
    stocks   = get_all_stocks(as_of=virtual_dt)   # 가상 시점 재고 계산
    available = [
        (pid, p, stocks[pid])
        for pid, p in products.items()
        if stocks.get(pid, 0) >= 1 and parse_dt(p["exp_date"]) >= virtual_dt
    ]
    available.sort(key=lambda x: x[0])
    print("\n[판매 가능한 재고 목록]")
    print(f"{'상품ID':<8} {'상품명':<20} {'수량'}")
    print("-" * 36)
    for pid, p, qty in available:
        print(f"{pid:<8} {p['name']:<20} {qty}")
    print(f"\n총 상품 개수: {len(available)}개")

# ══════════════════════════════════════════════════════════════
#  ENTRY POINT
# ══════════════════════════════════════════════════════════════
def main():
    global current_datetime
    startup_banner()                              # 6.1.1
    check_files()                                 # 6.1.2
    check_integrity()                             # 6.1.3
    current_datetime = input_current_datetime()   # 6.1.4
    auto_dispose(current_datetime)                # 6.1.5
    main_menu()                                   # 6.2

if __name__ == "__main__":
    main()

'스마트 무인 편의점' 재고 관리 프로그램 v1.0
무결성 검사 중… 완료

현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):
> 2026-04-17 13:44:00

기준 시각이 설정되었습니다.
총 1개의 재고가 유통기한 경과로 자동 폐기 처리되었습니다.

1. 고객 관리
2. 재고 관리
3. 판매 처리
4. 가상의 일시 기준 재고 조회
0. 종료
메뉴 > 3

[판매 처리]
상품ID 입력 (예: 1001,1002,1003): 1001, 1002. 1001
존재하지 않는 상품: 1001

현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):
> 2026-04-17 13:44:00

현재 일시는 기존 로그의 마지막 시각보다 늦어야 합니다.

현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):
> 2026-04-17 13:45:00

기준 시각이 설정되었습니다.
현재 폐기 처리할 유통기한 경과 상품이 없습니다.

1. 고객 관리
2. 재고 관리
3. 판매 처리
4. 가상의 일시 기준 재고 조회
0. 종료
메뉴 > 3

[판매 처리]
상품ID 입력 (예: 1001,1002,1003): 0004, 0006, 0004
같은 상품ID를 중복 입력할 수 없습니다.

현재 일시를 입력하세요 (YYYY-MM-DD HH:MM:SS):
> 2026-04-17 13:45:45

기준 시각이 설정되었습니다.
현재 폐기 처리할 유통기한 경과 상품이 없습니다.

1. 고객 관리
2. 재고 관리
3. 판매 처리
4. 가상의 일시 기준 재고 조회
0. 종료
메뉴 > 3

[판매 처리]
상품ID 입력 (예: 1001,1002,1003): 0004, 0006


KeyboardInterrupt: Interrupted by user

In [ ]:
from google.colab import drive
drive.mount('/content/drive')